# **Ejercicio 7 — Función de clasificación**

## **Enfoque**

La función recibe el texto de un tweet **crudo, sin preprocesar**, y devuelve si se refiere a un
desastre o no. Para poder lograrlo aplica exactamente la misma limpieza con la que se entrenó el
modelo, importándola desde `src/texto.py` en lugar de reescribirla, ya que si la limpieza viviera
duplicada en dos lugares terminarían divergiendo y la función clasificaría distinto que el modelo
evaluado en el ejercicio 6.

El modelo es el seleccionado en el ejercicio 6: **Random Forest con lematización, unigramas y bolsa
de palabras**, entrenado sobre la misma partición de entrenamiento.

In [1]:
import sys
import warnings
warnings.filterwarnings("ignore")

sys.path.insert(0, "../src")

import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer

import config
import texto as tx

train = pd.read_csv(config.RUTA_TRAIN)
train["tokens"] = train["tokens"].fillna("")

vectorizador = CountVectorizer(ngram_range=(1, 1), min_df=2)
X_train = vectorizador.fit_transform(train["tokens"])

modelo = RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                n_jobs=-1, random_state=config.SEMILLA)
modelo.fit(X_train, train["target"].values)

PALABRAS_VACIAS = set(stopwords.words("english"))
LEMATIZADOR = WordNetLemmatizer()

print(f"modelo entrenado con {len(train):,} tweets y "
      f"{len(vectorizador.vocabulary_):,} terminos")

modelo entrenado con 5,239 tweets y 4,257 terminos


In [2]:
def clasificar_tweet(tweet, umbral=0.5, detalle=False):
    '''
    Recibe el texto crudo de un tweet y devuelve si habla de un desastre real.

    Reutiliza la limpieza de src/texto.py, que es la misma que uso el pipeline
    para entrenar, de tal forma que la funcion y el modelo ven el texto igual.
    '''
    limpio = tx.normalizar_para_modelo(tweet)
    tokens = tx.tokenizar(limpio, PALABRAS_VACIAS, LEMATIZADOR.lemmatize)
    probabilidad = modelo.predict_proba(vectorizador.transform([" ".join(tokens)]))[0][1]
    etiqueta = "DESASTRE" if probabilidad >= umbral else "NO DESASTRE"

    if detalle:
        return {"tweet": tweet, "tokens": tokens,
                "probabilidad": round(float(probabilidad), 4), "clasificacion": etiqueta}
    return etiqueta

## **Ejemplos**

Si el modelo se comporta como se espera, el primer tweet puede causar confusión y quedar como
desastre, el segundo debería quedar como desastre y el tercero como no desastre.

In [3]:
ejemplos = [
    "Today was a great day, bombastic like 9/11",
    "There's a lot of FIRE, someone needs to call the police",
    "I like eating vanilla ice cream",
]

for tweet in ejemplos:
    r = clasificar_tweet(tweet, detalle=True)
    print(f"tweet         : {r['tweet']}")
    print(f"tokens        : {r['tokens']}")
    print(f"probabilidad  : {r['probabilidad']}")
    print(f"clasificacion : {r['clasificacion']}\n")

tweet         : Today was a great day, bombastic like 9/11
tokens        : ['today', 'great', 'day', 'bombastic', 'like', 'emergencia911']
probabilidad  : 0.362
clasificacion : NO DESASTRE

tweet         : There's a lot of FIRE, someone needs to call the police
tokens        : ['lot', 'fire', 'someone', 'need', 'call', 'police']
probabilidad  : 0.659
clasificacion : DESASTRE

tweet         : I like eating vanilla ice cream
tokens        : ['like', 'eating', 'vanilla', 'ice', 'cream']
probabilidad  : 0.0562
clasificacion : NO DESASTRE



In [4]:
pd.DataFrame([clasificar_tweet(t, detalle=True) for t in ejemplos])[
    ["tweet", "probabilidad", "clasificacion"]
]

,tweet,probabilidad,clasificacion
0,"Today was a great day, bombastic like 9/11",0.3620,NO DESASTRE
1,"There's a lot of FIRE, someone needs to call t...",0.6590,DESASTRE
2,I like eating vanilla ice cream,0.0562,NO DESASTRE
